# CWSB Phase 5 — ONNX export + pure-ONNX inference (one-run Colab)

Converts `contender_B.pt` (ConvNeXt-Tiny classifier) to a single self-contained
`cwsb_v1.onnx`, verifies **4-decimal parity** against PyTorch, then tests it on your
own photos with evidence heatmaps — using **only ONNX Runtime** for inference,
the way the Raspberry Pi / API will run it.

The ONNX graph contains: `/255` → ImageNet normalize → ConvNeXt → softmax, plus the
12×12 feature map as a second output for heatmaps. Resize + center-crop stay **outside**
the graph in Python, using the exact same ops as training (`eval_tf`) — baking a dynamic
resize into the traced graph distorts non-16:9 photos and changes scores (see the note
at the bottom).

No GPU needed — CPU runtime is fine. Run all cells top to bottom.

In [ ]:
# Cell 1 - install + upload the checkpoint
%pip -q install timm onnx onnxruntime onnxscript
import os, torch, timm, numpy as np
import torch.nn as nn, torch.nn.functional as F
from google.colab import files

pt_filename = 'contender_B.pt'
if not os.path.exists(pt_filename):
    print('Select and upload contender_B.pt (106 MB):')
    uploaded = files.upload()
    pt_filename = list(uploaded.keys())[0]
print('checkpoint file:', pt_filename)

In [ ]:
# Cell 2 - rebuild the exact architecture and load the weights
MODEL_NAME = 'convnext_tiny.fb_in22k_ft_in1k_384'
IMG = 384
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=2)
# weights_only=False: the checkpoint dict holds a numpy float (val_pr_auc),
# which the torch>=2.6 safe loader rejects. It is our own trusted file.
ckpt = torch.load(pt_filename, map_location='cpu', weights_only=False)
state_dict = ckpt.get('model', ckpt.get('state_dict', ckpt)) if isinstance(ckpt, dict) else ckpt
state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
model.load_state_dict(state_dict, strict=True)
model.eval()
if isinstance(ckpt, dict) and 'val_pr_auc' in ckpt:
    print(f"loaded: epoch {ckpt.get('epoch')}  val PR-AUC {float(ckpt['val_pr_auc']):.4f}")
print('PyTorch weights loaded.')

In [ ]:
# Cell 3 - wrap (normalize + model + softmax + feature map) and export to one .onnx file
class CwsbOnnx(nn.Module):
    """Input:  one 384x384x3 RGB uint8 image (already resized+center-cropped).
    Output: probabilities (1,2) = [healthy, wsb]  and  feature_map (1,768,12,12)."""
    def __init__(self, core):
        super().__init__()
        self.core = core
        self.register_buffer('mean', torch.tensor(MEAN).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(STD).view(1, 3, 1, 1))

    def forward(self, rgb_uint8):
        x = rgb_uint8.permute(2, 0, 1).unsqueeze(0).float() / 255.0  # == ToTensor()
        x = (x - self.mean) / self.std                               # == Normalize()
        feat = self.core.forward_features(x)                          # (1,768,12,12)
        logits = self.core.forward_head(feat)                         # (1,2)
        return F.softmax(logits, dim=1), feat

wrapped = CwsbOnnx(model).eval()
onnx_filename = 'cwsb_v1.onnx'
dummy = torch.randint(0, 256, (IMG, IMG, 3), dtype=torch.uint8)

torch.onnx.export(
    wrapped, dummy, onnx_filename,
    input_names=['rgb_384'],
    output_names=['probabilities', 'feature_map'],
    opset_version=17,
    dynamo=False,  # single self-contained file, no external .data
)
print(f'exported {onnx_filename}  ({os.path.getsize(onnx_filename)/1e6:.1f} MB)')

In [ ]:
# Cell 4 - 4-decimal parity check: full PyTorch eval path vs full ONNX path
import onnxruntime as ort
from PIL import Image, ImageOps
from torchvision import transforms

# the project's eval_tf, split into its PIL part and its tensor part
crop_tf = transforms.Compose([transforms.Resize(IMG + 32), transforms.CenterCrop(IMG)])
norm_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

def preprocess(pil_img):
    """EXIF-upright -> resize short side 416 -> center-crop 384 -> uint8 HWC.
    This is the ONLY code the Pi/API must replicate outside the .onnx file."""
    return np.asarray(crop_tf(pil_img.convert('RGB')), dtype=np.uint8)

session = ort.InferenceSession(onnx_filename, providers=['CPUExecutionProvider'])

worst = 0.0
for trial in range(5):  # five random 'photos' of odd sizes
    h, w = np.random.randint(500, 3000), np.random.randint(500, 3000)
    pil = Image.fromarray(np.random.randint(0, 256, (h, w, 3), dtype=np.uint8))
    crop = preprocess(pil)
    with torch.no_grad():  # PyTorch reference: the exact training-time eval path
        pt_probs = torch.softmax(model(norm_tf(Image.fromarray(crop)).unsqueeze(0)), 1).numpy()
    onnx_probs = session.run(['probabilities'], {'rgb_384': crop})[0]
    worst = max(worst, float(np.max(np.abs(pt_probs - onnx_probs))))

print(f'max |pytorch - onnx| over 5 random inputs: {worst:.2e}')
assert worst < 1e-4, f'parity FAILED ({worst})'
print('Parity check PASSED - matches to 4+ decimal places.')
files.download(onnx_filename)

In [ ]:
# Cell 5 - test on your own photos, PURE ONNX (upload a .zip of images, or single images)
import csv, glob, shutil, zipfile, cv2
import matplotlib.pyplot as plt

T_LOW, T_HIGH = 0.50, 0.90  # models/v1/config.json
BLUR_MIN, CLIPPED_MAX, DARK_MAX = 60.0, 0.20, 0.40  # same quality gate as the app

def quality_gate(pil):
    g = cv2.cvtColor(np.asarray(pil), cv2.COLOR_RGB2GRAY)
    s = 1024 / max(g.shape[1], 1)
    if s < 1:
        g = cv2.resize(g, (1024, max(int(g.shape[0] * s), 1)))
    if float(cv2.Laplacian(g, cv2.CV_64F).var()) < BLUR_MIN:
        return 'photo is blurry - hold steady and retake'
    if float((g > 250).mean()) > CLIPPED_MAX:
        return 'too much sun - shade the plant and retake'
    if float((g < 15).mean()) > DARK_MAX:
        return 'too dark - step back or use the flash'
    return None

def eigencam(feat_map):
    """Class-agnostic heatmap from the ONNX feature_map output (no gradients needed)."""
    fmap = feat_map[0]
    c, h, w = fmap.shape
    _, _, vt = np.linalg.svd(fmap.reshape(c, h * w), full_matrices=False)
    cam = vt[0].reshape(h, w)
    if np.corrcoef(cam.flatten(), fmap.mean(axis=0).flatten())[0, 1] < 0:
        cam = -cam
    cam = np.maximum(cam, 0)
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

# -- collect images --
extract_dir = 'uploaded_batch'
shutil.rmtree(extract_dir, ignore_errors=True)
os.makedirs(extract_dir)
print('Upload a .zip of photos, or individual image files:')
uploaded = files.upload()
for fname in uploaded:
    if fname.lower().endswith('.zip'):
        with zipfile.ZipFile(fname) as zf:
            zf.extractall(extract_dir)
    else:
        shutil.move(fname, os.path.join(extract_dir, fname))

VALID = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
image_paths = sorted(
    os.path.join(r, f) for r, _, fs in os.walk(extract_dir) for f in fs
    if f.lower().endswith(VALID) and not f.startswith('._') and '__MACOSX' not in r)
print(f'{len(image_paths)} images found\n')

# -- score with ONNX only --
out_dir = 'onnx_evidence'
shutil.rmtree(out_dir, ignore_errors=True)
os.makedirs(out_dir)
rows, saved = [['filename', 'wsb_score', 'verdict']], []

for img_path in image_paths:
    name = os.path.basename(img_path)
    pil = ImageOps.exif_transpose(Image.open(img_path)).convert('RGB')
    reject = quality_gate(pil)
    if reject:
        print(f'{name} -> GATE: {reject}')
        rows.append([name, '', 'INCONCLUSIVE - ' + reject])
        continue
    crop = preprocess(pil)                      # same function the parity check used
    probs, feat = session.run(None, {'rgb_384': crop})
    score = float(probs[0][1])
    verdict, color = (('WSB SIGNS LIKELY', 'red') if score >= T_HIGH else
                      ('NO VISIBLE WSB SIGNS', 'green') if score < T_LOW else
                      ('INCONCLUSIVE (retake)', 'orange'))
    print(f'{name} -> {score:.4f}  {verdict}')
    rows.append([name, f'{score:.4f}', verdict])

    # overlay on the 384x384 crop - the heatmap only describes what the model saw
    heat = cv2.applyColorMap(np.uint8(255 * cv2.resize(eigencam(feat), (IMG, IMG))), cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(crop, 0.55, cv2.cvtColor(heat, cv2.COLOR_BGR2RGB), 0.45, 0)
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(crop); ax[0].set_title('model input (384x384 crop)', fontsize=10)
    ax[1].imshow(overlay); ax[1].set_title(f'ONNX evidence\nscore {score:.4f} | {verdict}',
                                           color=color, fontsize=10, fontweight='bold')
    for a in ax: a.axis('off')
    fig.suptitle(name, fontsize=11)
    out = os.path.join(out_dir, f'evidence_{name}.png')
    plt.savefig(out, bbox_inches='tight', dpi=120)
    saved.append(out)
    plt.show(); plt.close(fig)

csv_path = os.path.join(out_dir, 'predictions_summary.csv')
with open(csv_path, 'w', newline='') as f:
    csv.writer(f).writerows(rows)
saved.append(csv_path)

with zipfile.ZipFile('onnx_results.zip', 'w') as zf:
    for p in saved:
        zf.write(p, arcname=os.path.basename(p))
print(f'\n{len(saved)-1} evidence images + CSV -> downloading onnx_results.zip')
files.download('onnx_results.zip')

## Notes — what was changed from the draft scripts, and why

1. **Dynamic any-resolution export removed.** With `dynamo=False` the export *traces* the model:
   `int(new_h)` inside `forward()` freezes the resize geometry of the 1080×1920 dummy into the
   graph, so photos with any other aspect ratio get distorted before scoring. The 720×1280 test
   passed only because it shares the 16:9 aspect. Also `F.interpolate` (no antialias) is not the
   same resize as training's `transforms.Resize`, which shifts scores. The fix: the graph takes a
   fixed 384×384×3 uint8 input and the 6-line `preprocess()` above is the only code the
   Pi/API replicates — using PIL, identical to `eval_tf`.
2. **`weights_only=False`** added to `torch.load` (checkpoint stores a numpy float; torch ≥ 2.6
   refuses it otherwise).
3. **Parity** now compares the full deployed path (crop → graph) against the full training-time
   eval path, on five random image sizes, not just bare logits on one tensor.
4. **Heatmap overlaid on the 384×384 crop**, not the full-resolution original — the feature map
   only describes the crop, so stretching it over the whole photo put heat on pixels the model
   never saw.
5. **Quality gate added** (blur / sun / dark → retake), same constants as `predict_folder.py`,
   so this behaves like the planned `/v1/predict` API.
6. EigenCAM is **class-agnostic** (shows where the model looked, not per-class evidence like
   true Grad-CAM). Good enough for field evidence; use `src/make_report_pdf.py` locally when you
   need real Grad-CAM.

**On the local machine afterwards:** put `cwsb_v1.onnx` in `models/v1/`, log the parity number in
`charan_working.md`, and tick Phase 5 in `NEXT_STEPS.md`.